In [ ]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

# ==========================================
# 1. REPRODUCTIBILITÉ & CONFIGURATION GENERALE
# ==========================================
# Définition d'une graine (seed) unique pour que chaque exécution
# génère exactement le même jeu de données.
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

# Définition des paramètres temporels de la simulation
# Fréquence IoT : 30 secondes (compromis POC réaliste)
FREQUENCE_IOT_SEC = 30
DATE_DEBUT = datetime(2024, 8, 1, 0, 0, 0)
DATE_FIN = datetime(2026, 8, 1, 0, 0, 0) # Simulation sur 3 mois complet

# Calcul de l'index temporel (la timeline globale commune)
delta_temps = DATE_FIN - DATE_DEBUT
total_intervalles = int(delta_temps.total_seconds() / FREQUENCE_IOT_SEC)
timestamps_globaux = [DATE_DEBUT + timedelta(seconds=i*FREQUENCE_IOT_SEC) for i in range(total_intervalles)]

print(f"--- Configuration Temporelle Initialisée ---")
print(f"Période : Du {DATE_DEBUT} au {DATE_FIN}")
print(f"Fréquence : Chaque {FREQUENCE_IOT_SEC} secondes")
print(f"Nombre de points temporels par machine : {len(timestamps_globaux):,}\n")

# ==========================================
# 2. RÉFÉRENTIEL MACHINES (Données Constantes)
# ==========================================
# Création d'un dictionnaire décrivant notre parc industriel (site unique en France)
referentiel_machines = {
    'MCH-001': {
        'nom': 'Centre Usinage 5-Axes (Aéro)',
        'usine': 'France_Main_Site',
        'age_initial_jours': 1200,          # Machine ancienne (sensible à l'usure mécanique)
        'date_installation': (DATE_DEBUT - timedelta(days=1200)).strftime('%Y-%m-%d')
    },
    'MCH-002': {
        'nom': 'Centre Usinage 3-Axes (Auto)',
        'usine': 'France_Main_Site',
        'age_initial_jours': 300,           # Machine récente (profil globalement sain)
        'date_installation': (DATE_DEBUT - timedelta(days=300)).strftime('%Y-%m-%d')
    },
    'MCH-003': {
        'nom': 'Robot Découpe & Laser (Mixte)',
        'usine': 'France_Main_Site',
        'age_initial_jours': 650,           # Machine d'âge moyen
        'date_installation': (DATE_DEBUT - timedelta(days=650)).strftime('%Y-%m-%d')
    },
    'MCH-004': {
        'nom': 'Centre Usinage 5-Axes (Aéro)',
        'usine': 'France_Main_Site',
        'age_initial_jours': 150,           # Neuve (jumelle de la MCH-001 mais sans l'usure de départ)
        'date_installation': (DATE_DEBUT - timedelta(days=150)).strftime('%Y-%m-%d')
    },
    'MCH-005': {
        'nom': 'Centre Usinage 3-Axes (Auto)',
        'usine': 'France_Main_Site',
        'age_initial_jours': 850,           # Ancienne (jumelle de la MCH-002, va permettre de comparer l'effet de l'âge sur le même type de process)
        'date_installation': (DATE_DEBUT - timedelta(days=850)).strftime('%Y-%m-%d')
    },
    'MCH-006': {
        'nom': 'Presse Hydraulique (Auto)',
        'usine': 'France_Main_Site',
        'age_initial_jours': 2100,          # Très vieille machine (idéale pour tester les fuites de pression fréquentes)
        'date_installation': (DATE_DEBUT - timedelta(days=2100)).strftime('%Y-%m-%d')
    },
    'MCH-007': {
        'nom': 'Robot Découpe Laser (Mixte)',
        'usine': 'France_Main_Site',
        'age_initial_jours': 400,           # Âge moyen
        'date_installation': (DATE_DEBUT - timedelta(days=400)).strftime('%Y-%m-%d')
    }
}

# Conversion en DataFrame Pandas pour une visualisation propre et manipulation future
df_referentiel = pd.DataFrame.from_dict(referentiel_machines, orient='index').reset_index()
df_referentiel.rename(columns={'index': 'machine_id'}, inplace=True)

print("--- Référentiel des Machines (Données Constantes) ---")
print(df_referentiel.to_string(index=False))

In [ ]:
# --- CONFIGURATION NOMINALE SPÉCIFIQUE PAR MACHINE ---
# Ici, on définit la signature physique de base (hors effet de l'âge) pour chaque machine
machines_config = {
    'MCH-001': {'type': 'CNC_5_Axes',  'secteur': 'Aéro', 'v_rot_titane': 1200, 'v_rot_alu': 6000, 'c_base_titane': 45.0, 'c_base_alu': 20.0, 't_base': 40.0, 'temps_cycle_titane': 420.0, 'temps_cycle_alu': 180.0, 'temps_cycle_acier': None},
    'MCH-004': {'type': 'CNC_5_Axes',  'secteur': 'Aéro', 'v_rot_titane': 1250, 'v_rot_alu': 6200, 'c_base_titane': 43.0, 'c_base_alu': 19.0, 't_base': 38.0, 'temps_cycle_titane': 400.0, 'temps_cycle_alu': 170.0, 'temps_cycle_acier': None}, # Légère variante de modèle
    'MCH-002': {'type': 'CNC_3_Axes',  'secteur': 'Auto', 'v_rot_titane': 0, 'v_rot_alu': 4500, 'c_base_titane': 0.0, 'c_base_alu': 22.0, 't_base': 35.0, 'temps_cycle_titane': None, 'temps_cycle_alu': 90.0, 'temps_cycle_acier': None}, # 100% Alu
    'MCH-005': {'type': 'CNC_3_Axes',  'secteur': 'Auto', 'v_rot_titane': 0, 'v_rot_alu': 4300, 'c_base_titane': 0.0, 'c_base_alu': 24.0, 't_base': 36.0, 'temps_cycle_titane': None, 'temps_cycle_alu': 95.0, 'temps_cycle_acier': None}, # Variante modèle
    'MCH-003': {'type': 'Robot_Laser', 'secteur': 'Mixte', 'v_rot_titane': 3000, 'v_rot_alu': 5000, 'c_base_titane': 35.0, 'c_base_alu': 15.0, 't_base': 34.0, 'temps_cycle_titane': 60.0, 'temps_cycle_alu': 40.0, 'temps_cycle_acier': None},
    'MCH-007': {'type': 'Robot_Laser', 'secteur': 'Mixte', 'v_rot_titane': 2800, 'v_rot_alu': 4800, 'c_base_titane': 37.0, 'c_base_alu': 16.0, 't_base': 33.0, 'temps_cycle_titane': 65.0, 'temps_cycle_alu': 42.0, 'temps_cycle_acier': None},
    'MCH-006': {'type': 'Presse_Hydr',  'secteur': 'Auto', 'v_rot_titane': 0, 'v_rot_alu': 0, 'c_base_titane': 0.0, 'c_base_alu': 0.0, 't_base': 45.0, 'temps_cycle_titane': None, 'temps_cycle_alu': None, 'temps_cycle_acier': 25.0}  # Ne tourne pas, gérée à part
}

print("--- Génération du Squelette Nominale Spécifique par Machine ---")

liste_dfs_machines = []

for mch_id, cfg in machines_config.items():
    print(f"Planification des profils nominaux pour la machine {mch_id} ({cfg['type']})...")

    statuts = []
    matériaux = []
    cadences_rpm = []
    courants_base = []
    pressions_base = []

    intervalle_lot = 480 # Lots de 4 heures
    compteur_intervalle = 0
    matériau_actuel = None

    for ts in timestamps_globaux:
        jour_semaine = ts.weekday()
        est_week_end = jour_semaine >= 5

        if est_week_end:
            statuts.append('Arrêt Opérateur')
            matériaux.append('Aucun')
            cadences_rpm.append(0)
            courants_base.append(1.5) # Courant de veille
            pressions_base.append(15.0 if cfg['type'] == 'Presse_Hydr' else 0.0)
        else:
            statuts.append('Production')

            # Gestion du changement de lot de matériau
            if compteur_intervalle % intervalle_lot == 0:
                if cfg['secteur'] == 'Aéro':
                    matériau_actuel = 'Titane' if random.random() < 0.60 else 'Aluminium'
                elif cfg['secteur'] == 'Mixte':
                    matériau_actuel = 'Titane' if random.random() < 0.50 else 'Aluminium'
                elif cfg['secteur'] == 'Auto':
                    matériau_actuel = 'Aluminium'
                compteur_intervalle = 0

            compteur_intervalle += 1

            if cfg['type'] == 'Presse_Hydr':
                matériaux.append('Acier') # Spécificité de la presse
                cadences_rpm.append(0)
                courants_base.append(55.0) # Pompe hydraulique nominale
                pressions_base.append(180.0)
            else:
                matériaux.append(matériau_actuel)
                pressions_base.append(0.0)

                # Application de la valeur nominale PROPRE à cette machine et au matériau
                if matériau_actuel == 'Titane':
                    cadences_rpm.append(cfg['v_rot_titane'])
                    courants_base.append(cfg['c_base_titane'])
                else: # Aluminium
                    cadences_rpm.append(cfg['v_rot_alu'])
                    courants_base.append(cfg['c_base_alu'])

    # Assemblage du DataFrame de la machine
    df_mch = pd.DataFrame({
        'timestamp': timestamps_globaux,
        'machine_id': mch_id,
        'type_machine': cfg['type'],
        'secteur': cfg['secteur'],
        'type_metal': matériaux,
        'vitesse_rotation_nominal': cadences_rpm,
        'courant_moteur_nominal': courants_base,
        'pression_hydraulique_nominal': pressions_base,
        'statut_nominal': statuts,
        'temp_base_moteur': cfg['t_base']
    })

    liste_dfs_machines.append(df_mch)

df_nominal_global = pd.concat(liste_dfs_machines, ignore_index=True)
print(f"\nSquelette nominal global généré ! Taille : {df_nominal_global.shape[0]:,} lignes.")

In [ ]:
# ==========================================
# 1bis. REGIME DE CADENCE DE PRODUCTION (Carnet de commandes simule)
# ==========================================
# Objectif : simuler des periodes de haute cadence, de cadence reduite,
# et des periodes ou la machine est presente (jour ouvre) mais non
# sollicitee faute de commandes ("Sous_Charge").
# Le regime change par blocs hebdomadaires (persistance realiste),
# avec une dynamique differente selon le secteur (Auto = demande plus
# cyclique / Aero-Mixte = demande plus stable).

print("--- Génération du Régime de Cadence de Production (Étape 1bis) ---")

FACTEUR_CADENCE = {
    'Haute_Cadence': 1.15,
    'Cadence_Nominale': 1.0,
    'Cadence_Reduite': 0.65,
    'Sous_Charge': 0.0,
}

nb_jours_total = (DATE_FIN - DATE_DEBUT).days
nb_semaines_total = nb_jours_total // 7 + 2

def generer_sequence_regime(secteur):
    regime = 'Cadence_Nominale'
    sequence = []
    for _ in range(nb_semaines_total):
        tirage = random.random()
        if secteur == 'Auto':
            # Demande plus cyclique / "bursty" (à-coups de commandes automobiles)
            if tirage < 0.15:
                regime = 'Haute_Cadence'
            elif tirage < 0.30:
                regime = 'Cadence_Reduite'
            elif tirage < 0.35:
                regime = 'Sous_Charge'
            else:
                regime = 'Cadence_Nominale'
        else:
            # Aéro / Mixte : demande plus stable, moins de trous de charge
            if tirage < 0.10:
                regime = 'Haute_Cadence'
            elif tirage < 0.20:
                regime = 'Cadence_Reduite'
            elif tirage < 0.23:
                regime = 'Sous_Charge'
            else:
                regime = 'Cadence_Nominale'
        sequence.append(regime)
    return sequence

liste_regimes_par_machine = {
    mch_id: generer_sequence_regime(cfg['secteur'])
    for mch_id, cfg in machines_config.items()
}

# --- Association vectorisée (machine, semaine) -> régime, via merge (rapide sur ~1M+ lignes) ---
lignes_correspondance = []
for mch_id, sequence in liste_regimes_par_machine.items():
    for sem_idx, regime in enumerate(sequence):
        lignes_correspondance.append((mch_id, sem_idx, regime))
df_correspondance = pd.DataFrame(lignes_correspondance, columns=['machine_id', 'semaine_indice', 'regime_cadence'])

df_nominal_global['jour_indice'] = (df_nominal_global['timestamp'] - DATE_DEBUT).dt.days
df_nominal_global['semaine_indice'] = df_nominal_global['jour_indice'] // 7

df_nominal_global = df_nominal_global.merge(df_correspondance, on=['machine_id', 'semaine_indice'], how='left')
df_nominal_global.drop(columns=['jour_indice', 'semaine_indice'], inplace=True)

# Le week-end reste un régime à part (aucune notion de cadence, machine fermée)
df_nominal_global.loc[df_nominal_global['statut_nominal'] == 'Arrêt Opérateur', 'regime_cadence'] = 'Repos_Hebdo'

df_nominal_global['facteur_cadence'] = df_nominal_global['regime_cadence'].map(FACTEUR_CADENCE).fillna(1.0)

# S assurer que ces colonnes sont bien en float (elles ont ete creees avec des int)
# avant toute multiplication par un facteur de cadence (evite les erreurs de dtype pandas).
df_nominal_global['vitesse_rotation_nominal'] = df_nominal_global['vitesse_rotation_nominal'].astype(float)
df_nominal_global['courant_moteur_nominal'] = df_nominal_global['courant_moteur_nominal'].astype(float)
df_nominal_global['pression_hydraulique_nominal'] = df_nominal_global['pression_hydraulique_nominal'].astype(float)

# --- Application du régime "Sous_Charge" : jour ouvré mais machine non sollicitée ---
masque_sous_charge = (df_nominal_global['regime_cadence'] == 'Sous_Charge') & (df_nominal_global['statut_nominal'] == 'Production')

df_nominal_global.loc[masque_sous_charge, 'statut_nominal'] = 'Arrêt Sous-Charge'
df_nominal_global.loc[masque_sous_charge, 'type_metal'] = 'Aucun'
df_nominal_global.loc[masque_sous_charge, 'vitesse_rotation_nominal'] = 0
df_nominal_global.loc[masque_sous_charge, 'courant_moteur_nominal'] = 1.5
df_nominal_global.loc[masque_sous_charge, 'pression_hydraulique_nominal'] = np.where(
    df_nominal_global.loc[masque_sous_charge, 'type_machine'] == 'Presse_Hydr', 15.0, 0.0
)

# --- Application du facteur de cadence sur les jours réellement en production ---
masque_prod_active = df_nominal_global['statut_nominal'] == 'Production'
df_nominal_global.loc[masque_prod_active, 'vitesse_rotation_nominal'] = (
    df_nominal_global.loc[masque_prod_active, 'vitesse_rotation_nominal'] * df_nominal_global.loc[masque_prod_active, 'facteur_cadence']
)
df_nominal_global.loc[masque_prod_active, 'courant_moteur_nominal'] = (
    df_nominal_global.loc[masque_prod_active, 'courant_moteur_nominal'] * (0.85 + 0.15 * df_nominal_global.loc[masque_prod_active, 'facteur_cadence'])
)

print("✅ Régimes de cadence générés et appliqués.")
print(df_nominal_global['regime_cadence'].value_counts())


In [ ]:
df_nominal_global

In [ ]:
print("--- Lancement du Moteur Physique (Étape 3) ---")

# Configuration de base (les moyennes autour desquelles on va faire varier)
CONFIG_PANNES = {
    'P1': {'nom': 'Usure outil', 'urgence': 2, 'mu_repar_min': 30, 'sigma_repar_min': 5, 'delai_alerte_min': 180, 'delai_survie_max_h': 12},
    'P2': {'nom': 'Casse brutale outil', 'urgence': 1, 'mu_repar_min': 60, 'sigma_repar_min': 10, 'delai_alerte_min': 0, 'delai_survie_max_h': 0},
    'P3': {'nom': 'Echauffement roulements', 'urgence': 2, 'mu_repar_min': 480, 'sigma_repar_min': 60, 'delai_alerte_min': 240, 'delai_survie_max_h': 8},
    'P4': {'nom': 'Desalignement axe', 'urgence': 2, 'mu_repar_min': 240, 'sigma_repar_min': 30, 'delai_alerte_min': 720, 'delai_survie_max_h': 48},
    'P5': {'nom': 'Grippage verin', 'urgence': 2, 'mu_repar_min': 120, 'sigma_repar_min': 20, 'delai_alerte_min': 360, 'delai_survie_max_h': 24},
    'P6': {'nom': 'Fuite hydraulique', 'urgence': 2, 'mu_repar_min': 180, 'sigma_repar_min': 40, 'delai_alerte_min': 120, 'delai_survie_max_h': 6},
    'P7': {'nom': 'Jeu mecanique', 'urgence': 3, 'mu_repar_min': 60, 'sigma_repar_min': 15, 'delai_alerte_min': 1440, 'delai_survie_max_h': 168},
    'P8': {'nom': 'Defaut lubrification', 'urgence': 2, 'mu_repar_min': 90, 'sigma_repar_min': 15, 'delai_alerte_min': 60, 'delai_survie_max_h': 4},
    'P9': {'nom': 'Surcharge electrique', 'urgence': 1, 'mu_repar_min': 15, 'sigma_repar_min': 3, 'delai_alerte_min': 0, 'delai_survie_max_h': 0},
    'P10': {'nom': 'Erreur soft Automate', 'urgence': 1, 'mu_repar_min': 20, 'sigma_repar_min': 5, 'delai_alerte_min': 0, 'delai_survie_max_h': 0}
}

liste_dfs_reels = []
PAS_PREVENTIF = 40320 # Fréquence des maintenance préventives fixée à 2 semaines
TAUX_OUBLI_PREVENTIF = 0.25 # 25% de chance que la maintenance ne soit pas faite

for mch_id in df_nominal_global['machine_id'].unique():
    df_mch = df_nominal_global[df_nominal_global['machine_id'] == mch_id].copy().sort_values('timestamp')
    n_lignes = len(df_mch)

    # 1. PRÉ-ALLOCATION MÉMOIRE NEUTRE
    temp_reelle = np.zeros(n_lignes)
    vib_rms = np.zeros(n_lignes)
    vib_peak = np.zeros(n_lignes)
    charge_moteur = np.zeros(n_lignes)
    age_reel_vecteur = np.zeros(n_lignes)
    age_virtuel_vecteur = np.zeros(n_lignes)
    temps_cycle_vecteur = np.full(n_lignes, np.nan)
    pieces_intervalle_vecteur = np.zeros(n_lignes, dtype=int)
    pieces_cumulees_vecteur = np.zeros(n_lignes, dtype=int)
    observation_operateur_vecteur = np.array([None] * n_lignes, dtype=object)

    vitesse_reelle = df_mch['vitesse_rotation_nominal'].values.astype(float)
    courant_reel = df_mch['courant_moteur_nominal'].values.astype(float)
    pression_reelle = df_mch['pression_hydraulique_nominal'].values.astype(float)
    statut_reel = df_mch['statut_nominal'].values.astype(object)

    label_gmao = np.ones(n_lignes, dtype=object) * 'Sain'
    historique_arrets_gmao = []

    # Initialisation réaliste du passé (70% de l'âge réel)
    age_reel_jours = referentiel_machines[mch_id]['age_initial_jours']
    age_virtuel_jours = age_reel_jours * 0.70

    pannes_latentes = set()
    pannes_alerte_mesurees = set()
    delai_avant_alerte = {}
    delai_survie_apres_alerte = {}

    temps_arret_restant = 0
    type_arret_actif = None
    panne_en_cause = None

    # Compteurs de production (temps de cycle / pièces produites)
    bucket_pieces = 0.0
    compteur_pieces_total = 0

    # Mapping des observations opérateur possibles par panne latente
    OBS_PAR_PANNE = {
        'P1': 'Bruit de coupe anormal',
        'P2': 'Bruit sec / choc violent',
        'P3': 'Odeur de surchauffe',
        'P4': 'Vibrations ressenties au poste',
        'P5': 'Mouvement irrégulier du vérin',
        'P6': "Trace d'huile / fuite visible",
        'P7': 'Jeu mécanique perceptible',
        'P8': 'Bruit de frottement sec',
        'P9': "Odeur de brûlé / disjoncteur",
        'P10': 'Ecran automate figé',
    }
    PROBA_SIGNALEMENT = {
        'P1': 0.003, 'P2': 0.02, 'P3': 0.004, 'P4': 0.006,
        'P5': 0.004, 'P6': 0.006, 'P7': 0.003, 'P8': 0.004,
        'P9': 0.01, 'P10': 0.002,
    }
    # Probabilite d'un signalement PRECOCE (pendant l'incubation, avant l'alerte
    # systeme officielle) : calibree pour que la probabilite CUMULEE sur toute
    # la duree d'incubation (delai_alerte_min) reste ~10-15%, quelle que soit
    # sa longueur (1h pour P8 a 24h pour P7). Sans ce calibrage, une simple
    # reprise de PROBA_SIGNALEMENT sur pannes_latentes rendrait le signalement
    # quasi certain pour les pannes a incubation longue (ex. P7).
    PROBA_SIGNALEMENT_PRECOCE = {
        'P1': 0.0003, 'P3': 0.00025, 'P4': 0.00008,
        'P5': 0.00017, 'P6': 0.0005, 'P7': 0.00004, 'P8': 0.001,
    }
    OBSERVATIONS_RAS = ['RAS - Bruit habituel', 'Confort poste signalé', 'Nettoyage effectué']

    # Seuil dynamique de déclenchement de l'effet domino (P8 -> P3)
    # De base 240 min (480 pas), réduit si la machine est vieille, allongé si elle est jeune
    # max(60,...) => 60 sert de valeur plancher (sécurité théorique dans le cas d'une machine très vielle)
    compteur_p8 = 0
    seuil_domino_p8_p3 = int(max(60, 240 * (1000.0 / age_virtuel_jours)) * 60 / FREQUENCE_IOT_SEC)

    # Signature vibratoire par type
    cfg_mch = machines_config[mch_id]
    if cfg_mch['type'] == 'CNC_5_Axes':
        vib_rms_base = 0.7; vib_peak_base = 1.1
    elif cfg_mch['type'] == 'CNC_3_Axes':
        vib_rms_base = 0.8; vib_peak_base = 1.3
    elif cfg_mch['type'] == 'Robot_Laser':
        vib_rms_base = 0.4; vib_peak_base = 0.6
    else:
        vib_rms_base = 1.5; vib_peak_base = 2.5

    facteur_vieillissement_0 = (age_virtuel_jours / 1000.0) ** 2.5
    age_reel_vecteur[0] = age_reel_jours
    age_virtuel_vecteur[0] = age_virtuel_jours
    temp_reelle[0] = cfg_mch['t_base'] + (facteur_vieillissement_0 * 4.0)
    vib_rms[0] = vib_rms_base + (facteur_vieillissement_0 * 0.25)
    vib_peak[0] = vib_peak_base + (facteur_vieillissement_0 * 0.25)

    print(f"Simulation physique stochastique : {mch_id} (Réel: {age_reel_jours:.1f}j, Virtuel: {age_virtuel_jours:.1f}j)...")

    # Usure reelle couplee materiau x cadence : le titane est plus abrasif que
    # l alu/acier, et une cadence soutenue use davantage qu une cadence reduite.
    # A l arret (week-end, sous-charge, maintenance), on garde une tres legere
    # usure calendaire residuelle (corrosion, graisse qui seche...) plutot qu un
    # gel total, plus realiste qu un vieillissement nul.
    FACTEUR_USURE_MATIERE = {'Titane': 1.8, 'Acier': 1.3, 'Aluminium': 1.0}
    FACTEUR_USURE_CALENDAIRE = 0.15

    # 2. BOUCLE TEMPORELLE
    for i in range(n_lignes):
        statut_nom = df_mch['statut_nominal'].iloc[i]
        metal = df_mch['type_metal'].iloc[i]
        type_mch = df_mch['type_machine'].iloc[i]
        facteur_cadence_jour = df_mch['facteur_cadence'].iloc[i]

        if i > 0:
            age_reel_jours += FREQUENCE_IOT_SEC / 86400.0
            if statut_nom == 'Production':
                facteur_usure_totale = FACTEUR_USURE_MATIERE.get(metal, 1.0) * max(facteur_cadence_jour, 0.01)
            else:
                facteur_usure_totale = FACTEUR_USURE_CALENDAIRE
            age_virtuel_jours += (FREQUENCE_IOT_SEC / 86400.0) * facteur_usure_totale

        age_reel_vecteur[i] = age_reel_jours
        age_virtuel_vecteur[i] = age_virtuel_jours

        facteur_vieillissement = (age_virtuel_jours / 1000.0) ** 2.5
        facteur_age_panne = 1.0 + (age_virtuel_jours / 300.0) ** 2
        degrade_fond_vib = facteur_vieillissement * 0.25
        degrade_fond_temp = facteur_vieillissement * 4.0

        if statut_nom in ('Arrêt Opérateur', 'Arrêt Sous-Charge'):
            statut_reel[i] = statut_nom
            temp_reelle[i] = max(22.0, temp_reelle[i-1] - 0.05) if i > 0 else cfg_mch['t_base']
            vib_rms[i] = 0.05
            vib_peak[i] = 0.1
            charge_moteur[i] = 0.0
            temps_cycle_vecteur[i] = np.nan
            pieces_intervalle_vecteur[i] = 0
            pieces_cumulees_vecteur[i] = compteur_pieces_total
            continue

        # --- GESTION DE LA MAINTENANCE (DURÉE VARIABLE) ---
        if temps_arret_restant > 0:
            statut_reel[i] = 'Maintenance'
            label_gmao[i] = f"Maintenance_{type_arret_actif}" + (f"_{panne_en_cause}" if panne_en_cause else "")
            vitesse_reelle[i] = 0.0
            courant_reel[i] = 0.4
            pression_reelle[i] = 10.0 if type_mch == 'Presse_Hydr' else 0.0
            temp_reelle[i] = max(22.0, temp_reelle[i-1] - 0.12) if i > 0 else cfg_mch['t_base']
            vib_rms[i] = 0.0
            vib_peak[i] = 0.0
            charge_moteur[i] = 0.0
            temps_cycle_vecteur[i] = np.nan
            pieces_intervalle_vecteur[i] = 0
            pieces_cumulees_vecteur[i] = compteur_pieces_total

            temps_arret_restant -= 1
            if temps_arret_restant == 0:
                # SORTIE DE MAINTENANCE : Application de la maintenance imparfaite stochastique
                if type_arret_actif == 'Preventif':
                    # Moyenne à 0.80 (20% de réduction), écart-type de 0.05
                    coeff_reset = np.random.normal(0.80, 0.05)
                    # Sécurisation pour éviter un coefficient aberrant (ex: supérieur à 1 ou trop bas)
                    coeff_reset = np.clip(coeff_reset, 0.70, 0.90)
                    age_virtuel_jours = max(age_reel_jours * 0.50, age_virtuel_jours * coeff_reset)

                elif type_arret_actif == 'Correctif':
                    # Moyenne à 0.90 (10% de réduction), écart-type de 0.08
                    coeff_reset = np.random.normal(0.90, 0.08)
                    coeff_reset = np.clip(coeff_reset, 0.85, 0.95)
                    age_virtuel_jours = max(age_reel_jours * 0.50, age_virtuel_jours * coeff_reset)

                # Reset complet des pannes latentes résolues
                pannes_latentes.clear()
                pannes_alerte_mesurees.clear()
                delai_avant_alerte.clear()
                delai_survie_apres_alerte.clear()
                panne_en_cause = None
                type_arret_actif = None
            continue

        # --- PLAN DE MAINTENANCE PRÉVENTIVE (AVEC DURÉE DE RÉPARATION VARIABLE) ---
        if i > 0 and i % PAS_PREVENTIF == 0:
            if random.random() > TAUX_OUBLI_PREVENTIF:
                statut_reel[i] = 'Maintenance'
                type_arret_actif = 'Preventif'
                panne_en_cause = 'PLANIFIEE'
                # Durée préventive nominale de 120 min, varie selon l'encombrement / loi normale
                duree_prev_aleatoire = max(60, int(np.random.normal(120, 15)))
                temps_arret_restant = int(duree_prev_aleatoire * 60 / FREQUENCE_IOT_SEC)
                historique_arrets_gmao.append(i)
                continue
            else:
                print(f"   ⚠️ [Négligence] Maintenance préventive sautée au pas {i} pour {mch_id} !")

        # --- EFFET DOMINO CONTEXTUEL AVEC INCERTITUDE (P8 -> P3) ---
        if 'P8' in pannes_latentes:
            compteur_p8 += 1
            # Si on dépasse le seuil dynamique, on n'applique pas un déclenchement automatique,
            # mais une probabilité qui augmente à chaque pas de temps (ici 0.5% de chance par pas de 30s)
            if compteur_p8 > seuil_domino_p8_p3 and 'P3' not in pannes_latentes:
                if random.random() < 0.005:  # Ajout de l'incertitude (1 chance sur 200 à chaque pas)
                    pannes_latentes.add('P3')

                    # Délai de base aléatoire (Loi normale autour de la configuration de P3)
                    mu_p3 = CONFIG_PANNES['P3']['delai_alerte_min']
                    variation_p3 = np.random.normal(mu_p3, mu_p3 * 0.15)

                    # Accélération contextuelle (âge virtuel + nombre de pannes déjà présentes)
                    nb_autres_pannes = len(pannes_latentes) - 1
                    facteur_acceleration = (1000.0 / age_virtuel_jours) * (1.0 / (1.0 + 0.3 * nb_autres_pannes))

                    delai_final_min = max(15, int(variation_p3 * min(1.0, facteur_acceleration)))
                    delai_avant_alerte['P3'] = int(delai_final_min * 60 / FREQUENCE_IOT_SEC)
        else:
            compteur_p8 = 0

        prob_casse_p2 = 0.000002 * facteur_age_panne * (6.0 if 'P4' in pannes_latentes else 1.0)

        # --- APPARITION DES PANNES LATENTES ET DÉLAIS STOCHASTIQUES CONTEXTUELS ---
        for p_id, cfg in CONFIG_PANNES.items():
            if p_id == 'P3' or (p_id == 'P6' and type_mch != 'Presse_Hydr'): continue
            if (p_id == 'P1' or p_id == 'P2' or p_id == 'P4') and type_mch == 'Presse_Hydr': continue

            if p_id not in pannes_latentes:
                prob = prob_casse_p2 if p_id == 'P2' else 0.000004 * facteur_age_panne
                if random.random() < prob:
                    pannes_latentes.add(p_id)

                    # LOGIQUE VARIABLE CONTEXTUELLE : Le temps d'incubation (avant alerte)
                    if cfg['delai_alerte_min'] > 0:
                        # Variation de base (Loi normale : +/- 15% d'écart type)
                        variation_aleatoire = np.random.normal(cfg['delai_alerte_min'], cfg['delai_alerte_min'] * 0.15)
                        # Multiplicateur contexte : Réduit par l'âge virtuel et par le cumul d'autres pannes
                        nb_autres_pannes = len(pannes_latentes) - 1
                        facteur_acceleration = (1000.0 / age_virtuel_jours) * (1.0 / (1.0 + 0.3 * nb_autres_pannes))

                        delai_final_min = max(10, int(variation_aleatoire * min(1.0, facteur_acceleration)))
                        delai_avant_alerte[p_id] = int(delai_final_min * 60 / FREQUENCE_IOT_SEC)
                    else:
                        delai_avant_alerte[p_id] = 0

        # --- PHYSIQUE DES CAPTEURS ---
        charge_base = 80.0 if metal == 'Titane' else (45.0 if metal == 'Aluminium' else 0.0)
        if type_mch == 'Presse_Hydr': charge_base = 70.0

        coeff_courant = 1.0
        coeff_vib = 1.0
        coeff_temp = 0.0

        if 'P1' in pannes_latentes: coeff_courant *= 1.25; coeff_vib *= 1.4
        if 'P3' in pannes_latentes: coeff_temp += 0.25
        if 'P4' in pannes_latentes: coeff_vib *= 3.5
        if 'P5' in pannes_latentes: coeff_courant *= 1.15
        if 'P6' in pannes_latentes and type_mch == 'Presse_Hydr': pression_reelle[i] -= 0.22
        if 'P7' in pannes_latentes: coeff_vib *= 1.8
        if 'P8' in pannes_latentes: coeff_temp += 0.15

        charge_moteur[i] = min(100.0, charge_base * coeff_courant)
        courant_reel[i] = courant_reel[i] * coeff_courant

        temp_cible = df_mch['temp_base_moteur'].iloc[i] + (18.0 if metal == 'Titane' else 6.0) + degrade_fond_temp
        temp_reelle[i] = temp_reelle[i-1] + (temp_cible - temp_reelle[i-1]) * 0.01 + coeff_temp if i > 0 else temp_cible

        vib_rms[i] = ((vib_rms_base + degrade_fond_vib) * coeff_vib) + np.random.normal(0, 0.03)
        coeff_peak_p7 = 2.0 if 'P7' in pannes_latentes else 1.0
        vib_peak[i] = ((vib_peak_base + degrade_fond_vib) * coeff_vib * coeff_peak_p7) + np.random.normal(0, 0.05)

        # --- TEMPS DE CYCLE REEL ET COMPTAGE DES PIECES PRODUITES ---
        if type_mch == 'Presse_Hydr':
            temps_cycle_base = cfg_mch['temps_cycle_acier']
        elif metal == 'Titane':
            temps_cycle_base = cfg_mch['temps_cycle_titane']
        else:
            temps_cycle_base = cfg_mch['temps_cycle_alu']

        # La degradation mecanique ralentit le cycle (compensations, controles renforces)
        facteur_ralentissement_usure = 1.0
        if 'P1' in pannes_latentes: facteur_ralentissement_usure *= 1.10
        if 'P4' in pannes_latentes: facteur_ralentissement_usure *= 1.25
        if 'P7' in pannes_latentes: facteur_ralentissement_usure *= 1.15
        if 'P5' in pannes_latentes: facteur_ralentissement_usure *= 1.20
        if 'P6' in pannes_latentes and type_mch == 'Presse_Hydr': facteur_ralentissement_usure *= 1.30
        if 'P8' in pannes_latentes: facteur_ralentissement_usure *= 1.12

        temps_cycle_reel = (temps_cycle_base / max(0.3, facteur_cadence_jour)) * facteur_ralentissement_usure
        temps_cycle_reel *= np.random.normal(1.0, 0.04)
        temps_cycle_reel = max(5.0, temps_cycle_reel)
        temps_cycle_vecteur[i] = temps_cycle_reel

        bucket_pieces += FREQUENCE_IOT_SEC / temps_cycle_reel
        pieces_ce_pas = 0
        while bucket_pieces >= 1.0:
            bucket_pieces -= 1.0
            pieces_ce_pas += 1
            compteur_pieces_total += 1
        pieces_intervalle_vecteur[i] = pieces_ce_pas
        pieces_cumulees_vecteur[i] = compteur_pieces_total

        # --- OBSERVATION QUALITATIVE OPERATEUR (rapportage sporadique et imparfait) ---
        observation_du_pas = None
        if random.random() < 0.0004:
            observation_du_pas = random.choice(OBSERVATIONS_RAS)

        for p in sorted(pannes_latentes):
            en_alerte = p in pannes_alerte_mesurees
            if en_alerte:
                proba_obs = PROBA_SIGNALEMENT.get(p, 0.002)
            else:
                # Phase d'incubation : signalement precoce possible mais rare
                proba_obs = PROBA_SIGNALEMENT_PRECOCE.get(p, 0.0001)
            if random.random() < proba_obs:
                observation_du_pas = OBS_PAR_PANNE.get(p, 'Anomalie signalée')
                break

        observation_operateur_vecteur[i] = observation_du_pas

        # --- DÉLAIS DE SURVIE DYNAMIQUES (PRODUCTION DÉGRADÉE) ---
        declencher_gmao_correctif = False

        for p in sorted(pannes_latentes):
            if p not in pannes_alerte_mesurees:
                if delai_avant_alerte[p] > 0:
                    delai_avant_alerte[p] -= 1
                else:
                    # Passage en phase ALERTE (Seuils franchis)
                    pannes_alerte_mesurees.add(p)
                    label_gmao[i] = f"Alerte_{p}"

                    # LOGIQUE VARIABLE CONTEXTUELLE : Le délai de survie avant l'arrêt brutal
                    if CONFIG_PANNES[p]['delai_survie_max_h'] > 0:
                        survie_nominale_h = random.uniform(1, CONFIG_PANNES[p]['delai_survie_max_h'])
                        # Plus la machine subit de charge et d'usure, plus elle casse vite en alerte
                        facteur_survie_contexte = (1000.0 / age_virtuel_jours) * (80.0 / max(40, charge_moteur[i]))
                        survie_finale_h = max(0.2, survie_nominale_h * min(1.0, facteur_survie_contexte))
                        delai_survie_apres_alerte[p] = int(survie_finale_h * 3600 / FREQUENCE_IOT_SEC)
                    else:
                        delai_survie_apres_alerte[p] = 0
            else:
                label_gmao[i] = f"Alerte_{p}"
                if delai_survie_apres_alerte[p] > 0:
                    delai_survie_apres_alerte[p] -= 1
                else:
                    declencher_gmao_correctif = True
                    panne_en_cause = p

        # Seuils physiques critiques absolus
        if temp_reelle[i] >= 98.0: declencher_gmao_correctif = True; panne_en_cause = 'P3'
        if type_mch == 'Presse_Hydr' and pression_reelle[i] <= 130.0: declencher_gmao_correctif = True; panne_en_cause = 'P6'

        # --- DÉCLENCHEMENT DE PANNE (DURÉE DE REPARATION DYNAMIQUE) ---
        if declencher_gmao_correctif:
            statut_reel[i] = 'Panne'
            type_arret_actif = 'Correctif'
            historique_arrets_gmao.append(i)

            if panne_en_cause == 'P2': vib_peak[i] = 15.0
            if panne_en_cause == 'P9': courant_reel[i] = 160.0

            # Durée de réparation variable (Loi Normale)
            cfg = CONFIG_PANNES[panne_en_cause]
            duree_rep = max(15, int(np.random.normal(cfg['mu_repar_min'], cfg['sigma_repar_min'])))
            temps_arret_restant = int(duree_rep * 60 / FREQUENCE_IOT_SEC)
        elif len(pannes_alerte_mesurees) == 0 and statut_reel[i] != 'Maintenance':
            statut_reel[i] = 'Production'

    # Intégration finale
    df_mch['statut_reel'] = statut_reel
    df_mch['label_gmao'] = label_gmao
    df_mch['vitesse_rotation_reelle'] = vitesse_reelle
    df_mch['courant_moteur_reel'] = courant_reel
    df_mch['pression_hydraulique_reelle'] = pression_reelle
    df_mch['temperature_reelle'] = temp_reelle
    df_mch['vibration_rms'] = np.abs(vib_rms)
    df_mch['vibration_peak'] = np.abs(vib_peak)
    df_mch['charge_moteur_pct'] = charge_moteur
    df_mch['age_jours'] = age_reel_vecteur
    df_mch['age_virtuel_jours'] = age_virtuel_vecteur
    df_mch['temps_cycle_sec'] = temps_cycle_vecteur
    df_mch['nb_pieces_intervalle'] = pieces_intervalle_vecteur
    df_mch['nb_pieces_cumule'] = pieces_cumulees_vecteur
    df_mch['observation_operateur'] = observation_operateur_vecteur

    # Cible RUL
    rul_jours = np.full(n_lignes, np.nan)
    idx_debut = 0

    if historique_arrets_gmao:
        for idx_arret in historique_arrets_gmao:
            indices_bloc = np.arange(idx_debut, idx_arret + 1)
            rul_jours[idx_debut : idx_arret + 1] = (idx_arret - indices_bloc) * FREQUENCE_IOT_SEC / 86400.0
            idx_debut = idx_arret + 1

    df_mch['RUL_jours'] = rul_jours
    liste_dfs_reels.append(df_mch)

df_reel_global = pd.concat(liste_dfs_reels, ignore_index=True)
print(f"\n--- Étape 3 Terminée avec Succès (Modèle Stochastique Contextuel Complet) ---")

In [ ]:
print("--- Lancement du Moteur de Perturbations IoT (Étape 4) ---")

# Copie profonde pour préserver le socle de l'Étape 3
df_iot = df_reel_global.copy()

# ----------------------------------------------------------------
# 1. INJECTION DU BRUIT ÉLECTRONIQUE (Bruit Gaussien permanent)
# ----------------------------------------------------------------
masque_prod = df_iot['statut_reel'].isin(['Production', 'Panne'])

df_iot.loc[masque_prod, 'temperature_reelle'] += np.random.normal(0, 0.25, size=masque_prod.sum())
df_iot.loc[masque_prod, 'courant_moteur_reel'] += np.random.normal(0, 0.15, size=masque_prod.sum())
df_iot.loc[masque_prod, 'vibration_rms'] += np.random.normal(0, 0.02, size=masque_prod.sum())
df_iot.loc[masque_prod, 'vibration_peak'] += np.random.normal(0, 0.04, size=masque_prod.sum())
df_iot.loc[masque_prod, 'pression_hydraulique_reelle'] += np.random.normal(0, 0.5, size=masque_prod.sum())

# Assurer la cohérence physique minimale
df_iot['vibration_rms'] = df_iot['vibration_rms'].clip(lower=0.0)
df_iot['vibration_peak'] = df_iot['vibration_peak'].clip(lower=0.0)
df_iot['temperature_reelle'] = df_iot['temperature_reelle'].clip(lower=0.0)
df_iot['courant_moteur_reel'] = df_iot['courant_moteur_reel'].clip(lower=0.0)
df_iot['pression_hydraulique_reelle'] = df_iot['pression_hydraulique_reelle'].clip(lower=0.0)

# ----------------------------------------------------------------
# 2. INJECTION DES CAPTEURS BLOQUÉS (Stuck-at Fault)
# ----------------------------------------------------------------
PROBA_BLOCAGE_PAS = 0.00005
colonnes_capteurs = ['temperature_reelle', 'vibration_rms', 'pression_hydraulique_reelle']

for mch_id in df_iot['machine_id'].unique():
    masque_mch = df_iot['machine_id'] == mch_id
    indices_mch = df_iot[masque_mch].index.tolist()

    capteur_bloque = None
    valeur_bloquee = None
    temps_blocage_restant = 0

    for idx in indices_mch:
        if temps_blocage_restant > 0:
            df_iot.at[idx, capteur_bloque] = valeur_bloquee
            temps_blocage_restant -= 1
            if temps_blocage_restant == 0:
                capteur_bloque = None
                valeur_bloquee = None
            continue

        if df_iot.at[idx, 'statut_reel'] == 'Production' and random.random() < PROBA_BLOCAGE_PAS:
            capteur_bloque = random.choice(colonnes_capteurs)
            valeur_bloquee = df_iot.at[idx, capteur_bloque]
            temps_blocage_restant = random.randint(10, 60)
            df_iot.at[idx, capteur_bloque] = valeur_bloquee

print("✅ Bruit électronique et pannes 'Stuck-at' injectés.")

# ----------------------------------------------------------------
# 3. INJECTION DE LA DÉRIVE DE CALIBRATION (Sensor Drift) AVEC RESET PRÉVENTIF
# ----------------------------------------------------------------
print("⚡ Calcul de la dérive de calibration dynamique...")

for mch_id in df_iot['machine_id'].unique():
    masque_mch = df_iot['machine_id'] == mch_id
    indices_mch = df_iot[masque_mch].index.tolist()

    derive_accumulee = 0.0
    # Taux de dérive par pas de temps (la dérive grandit très lentement à chaque ligne)
    taux_derive_par_pas = random.uniform(0.00005, 0.0002)

    for idx in indices_mch:
        statut = df_iot.at[idx, 'statut_reel']
        label = df_iot.at[idx, 'label_gmao']

        # Si une maintenance préventive vient de se terminer (ligne juste avant le retour en production)
        if label == 'Maintenance_Preventif_PLANIFIEE':
            # Reset de la dérive : l'étalonnage est fait !
            derive_accumulee = 0.0
        else:
            # La dérive augmente lentement, mais uniquement quand la machine vieillit (en prod ou panne)
            if statut in ['Production', 'Panne']:
                derive_accumulee += taux_derive_par_pas

        # On applique la dérive sur la température réelle
        df_iot.at[idx, 'temperature_reelle'] += derive_accumulee

print("✅ Dérive de calibration lente injectée (avec reset lors des maintenances préventives).")

# ----------------------------------------------------------------
# 4. INJECTION DE PICS ET VALEURS ABERRANTES (Spikes / Outliers)
# ----------------------------------------------------------------
# CORRECTION : Utilisation de 'statut_reel' (le nom de colonne actuel)
PROBA_SPIKE = 0.0002

for idx in df_iot[df_iot['statut_reel'] == 'Production'].index:
    if random.random() < PROBA_SPIKE:
        capteur_cible = random.choice(['courant_moteur_reel', 'vibration_peak', 'temperature_reelle'])

        if capteur_cible == 'courant_moteur_reel':
            df_iot.at[idx, capteur_cible] *= random.uniform(4.0, 8.0)
        elif capteur_cible == 'vibration_peak':
            df_iot.at[idx, capteur_cible] = random.uniform(20.0, 45.0)
        elif capteur_cible == 'temperature_reelle':
            df_iot.at[idx, capteur_cible] = random.uniform(180.0, 220.0)

print("✅ Pics de tension, chocs et anomalies thermiques ('Spikes') injectés.")

# ----------------------------------------------------------------
# 5. PERTE DE PAQUETS RÉSEAU (Data Dropping)
# ----------------------------------------------------------------
TAUX_PERTE_RESEAU = 0.02
masque_perte = np.random.rand(len(df_iot)) < TAUX_PERTE_RESEAU

colonnes_telemetrie = [
    'vitesse_rotation_reelle', 'courant_moteur_reel',
    'pression_hydraulique_reelle', 'temperature_reelle',
    'vibration_rms', 'vibration_peak', 'charge_moteur_pct'
]

df_iot.loc[masque_perte, colonnes_telemetrie] = np.nan
print(f"✅ Pertes réseau simulées : {masque_perte.sum():,} lignes contiennent des coupures (NaN).")

# ----------------------------------------------------------------
# 6. RENOMMAGE FINAL DES COLONNES (Couche IoT Brute)
# ----------------------------------------------------------------
df_iot = df_iot.rename(columns={
    'vitesse_rotation_reelle': 'iot_vitesse_rotation',
    'courant_moteur_reel': 'iot_courant_moteur',
    'pression_hydraulique_reelle': 'iot_pression_hydraulique',
    'temperature_reelle': 'iot_temperature',
    'vibration_rms': 'iot_vibration_rms',
    'vibration_peak': 'iot_vibration_peak',
    'charge_moteur_pct': 'iot_charge_moteur',
    'statut_reel': 'iot_statut_machine'
})

print(f"--- Étape 4 Terminée avec Succès | Dimensions du Dataset : {df_iot.shape[0]:,} lignes. ---")

In [ ]:
df_iot.describe()

In [ ]:
nom_fichier_csv = "dataset_brut.csv"

print(f"💾 Exportation en cours vers '{nom_fichier_csv}'...")
df_iot.to_csv(nom_fichier_csv, index=False, encoding='utf-8')
print("✅ Exportation terminée ! Le fichier est prêt.")